In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
np.random.seed(42)

In [2]:
ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "ml_df_v2.csv").exists() and (ROOT.parent / "data" / "ml_df_v2.csv").exists():
    ROOT = ROOT.parent

DATA = ROOT / "data"
ML_DF_V2_PATH = DATA / "ml_df_v2.csv"

if not ML_DF_V2_PATH.exists():
    raise FileNotFoundError("Run n3.1_feature_engineering.ipynb first to create ml_df_v2.csv")

In [3]:
ml_df = pd.read_csv(ML_DF_V2_PATH)
if "date" in ml_df.columns:
    ml_df["date"] = pd.to_datetime(ml_df["date"], errors="coerce")

print("ml_df_v2 shape:", ml_df.shape)
ml_df[["year", "home_team", "away_team", "host_country"]].head()

ml_df_v2 shape: (862, 183)


,year,home_team,away_team,host_country
0,1930,France,Mexico,Uruguay
1,1930,United States,Belgium,Uruguay
2,1930,Romania,Peru,Uruguay
3,1930,Yugoslavia,Brazil,Uruguay
4,1930,Argentina,France,Uruguay


## v4 Features: Neutral Venue and Home Advantage

In [4]:
# Map host_country values to team name format used in home_team/away_team
HOST_TO_TEAM = {
    "USA": "United States", "United States": "United States",
    "Korea Republic": "South Korea", "South Korea": "South Korea", "Korea DPR": "North Korea",
    "Côte d'Ivoire": "Ivory Coast", "Ivory Coast": "Ivory Coast",
}

def normalize_for_host(s):
    if pd.isna(s) or not str(s).strip():
        return ""
    t = str(s).strip()
    return HOST_TO_TEAM.get(t, t)

def is_host(team, host):
    if pd.isna(team) or pd.isna(host) or not str(host).strip():
        return False
    team_n = normalize_for_host(team)
    host_n = normalize_for_host(host)
    return team_n == host_n or str(team).strip() == str(host).strip()

In [5]:
# is_neutral_venue: 1 if neither team is host, else 0
# home_advantage_strength: 1 if home is host, 0 if neutral, -1 if away is host
home_is_host = ml_df.apply(lambda r: is_host(r["home_team"], r["host_country"]), axis=1)
away_is_host = ml_df.apply(lambda r: is_host(r["away_team"], r["host_country"]), axis=1)

ml_df_v4 = ml_df.copy()
ml_df_v4["is_neutral_venue"] = (~home_is_host & ~away_is_host).astype(int)
ml_df_v4["home_advantage_strength"] = np.where(home_is_host, 1, np.where(away_is_host, -1, 0))

print("v4 features added:")
print(ml_df_v4[["home_team", "away_team", "host_country", "is_neutral_venue", "home_advantage_strength"]].head(10))

v4 features added:
       home_team away_team host_country  is_neutral_venue  \
0         France    Mexico      Uruguay                 1   
1  United States   Belgium      Uruguay                 1   
2        Romania      Peru      Uruguay                 1   
3     Yugoslavia    Brazil      Uruguay                 1   
4      Argentina    France      Uruguay                 1   
5          Chile    Mexico      Uruguay                 1   
6  United States  Paraguay      Uruguay                 1   
7     Yugoslavia   Bolivia      Uruguay                 1   
8        Uruguay      Peru      Uruguay                 0   
9      Argentina    Mexico      Uruguay                 1   

   home_advantage_strength  
0                        0  
1                        0  
2                        0  
3                        0  
4                        0  
5                        0  
6                        0  
7                        0  
8                        1  
9                  

In [6]:
output_path = ROOT / "data" / "ml_df_v4.csv"
ml_df_v4.to_csv(output_path, index=False)
print("Saved:", output_path)
print("ml_df_v4 shape:", ml_df_v4.shape)

Saved: /Users/jeanphilippeauguste/Downloads/DS4-World-Cup-Project-Phase/data/ml_df_v4.csv
ml_df_v4 shape: (862, 185)
